# MetaGen Brain: Model Training & Fine-Tuning Suite

This notebook contains the foundational fine-tuning pipeline for the Mistral 7B Instruct YouTube SEO model.

### Lightweight Small Language Models (SLMs) for Sub-10s CPU Fallback:
For high-speed CPU deployment (under 10s inference latency on 2 vCPUs), train our specialized SLMs:
- **Qwen 2.5 1.5B Instruct:** notebooks/YT_Model_Brain_Qwen2.5_1.5B.ipynb (Native ChatML, ~1.05 GB GGUF, ~35 tok/s CPU)
- **Llama 3.2 1B Instruct:** notebooks/YT_Model_Brain_Llama3.2_1B.ipynb (Llama 3 formatting, ~750 MB GGUF, ~40 tok/s CPU)

### Full-Scale 7B/8B Parameter Models:
- **Qwen 2.5 7B Instruct:** notebooks/YT_Model_Brain_Qwen.ipynb
- **Llama 3.1 8B Instruct:** notebooks/YT_Model_Brain_Llama3.ipynb
- **Phi-4 Mini 3.8B:** notebooks/YT_Model_Brain_Phi4.ipynb


In [ ]:
!pip install -q -U transformers peft trl datasets bitsandbytes accelerate

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
import gc

In [ ]:
# Load your newly created dataset
dataset = load_dataset("json", data_files="youtube_finetuning_dataset.json", split="train")

def format_prompts(examples):
    """Formats the JSON into a single string for the model to read."""
    texts = []
    for instruction, input_text, output in zip(examples['instruction'], examples['input'], examples['output']):
        # Create the prompt template
        text = f"<s>[INST] {instruction}\n{input_text} [/INST] {output}</s>"
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(format_prompts, batched=True)

In [ ]:
# 1. Force clear any residual memory
torch.cuda.empty_cache()
gc.collect()

# 2. Configure 4-bit Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# 3. Load the Base Model and Tokenizer
model_id = "mistralai/Mistral-7B-Instruct-v0.2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

print("Downloading and loading model weights to GPU...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.float16
)

print("Preparing model for training...")
model = prepare_model_for_kbit_training(model)

# 4. Configure LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

# -----------------------------------------------------------------
# THE ULTIMATE BFLOAT16 PURGE
# -----------------------------------------------------------------
print("Hunting down and converting lingering BFloat16 weights...")
for name, param in model.named_parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float16)

# Force the internal config so the Trainer doesn't get confused
model.config.torch_dtype = torch.float16

print("✅ Model loaded and completely purged of BFloat16!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Preparing model for training...
Hunting down and converting lingering BFloat16 weights...
✅ Model loaded and completely purged of BFloat16!


In [ ]:
# 5. Set up the Trainer (UPDATED for the latest trl library)
training_args = SFTConfig(
    output_dir="./yt_lora_results",
    dataset_text_field="text",
    max_length=512,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=150,
    optim="paged_adamw_8bit",
    fp16=True,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
)

In [ ]:
# 6. START TRAINING!
# a. Force all trainable parameters (LoRA weights) to standard Float32
# This bypasses the T4 hardware crash and stabilizes the training loss
for param in trainer.model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

# b. Double-check and forcefully eliminate any deep-nested BFloat16 tensors
for name, param in trainer.model.named_parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float16)

# c. Explicitly hardcode the trainer's precision settings just before execution
trainer.args.bf16 = False
trainer.args.fp16 = True

# d. START TRAINING!
print("Starting the fine-tuning process...")
trainer.train()

Starting the fine-tuning process...


Step,Training Loss
10,1.825037
20,1.394576
30,1.326567
40,1.285037
50,1.330285
60,1.218434
70,1.183842


Step,Training Loss
10,1.825037
20,1.394576
30,1.326567
40,1.285037
50,1.330285
60,1.218434
70,1.183842
80,1.240662
90,1.219184
100,1.153101


TrainOutput(global_step=150, training_loss=1.2415373229980469, metrics={'train_runtime': 1692.9712, 'train_samples_per_second': 0.709, 'train_steps_per_second': 0.089, 'total_flos': 2.368789905363763e+16, 'train_loss': 1.2415373229980469})

In [ ]:
# 7. Save your custom weights
trainer.model.save_pretrained("yt-custom-lora-adapter")
print("Training complete! Custom weights saved.")

Training complete! Custom weights saved.


In [ ]:
import torch

# 1. Define a brand new topic to test
test_instruction = "You are an expert YouTube SEO strategist. Generate a viral title, description, and tags for this topic."
test_input = "Topic: How to build a modern portfolio website. Context: In this video, I will show you step by step how to build a stunning, responsive portfolio website using React, Tailwind CSS, and Neo-Brutalism design concepts to help you land your first tech job."

# 2. Format it exactly how the model was trained
prompt = f"<s>[INST] {test_instruction}\n{test_input} [/INST]"

# 3. Tokenize and move to GPU
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# 4. Generate the response
print("Generating AI response...\n")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=250,
        temperature=0.7,      # Add a little creativity
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

# 5. Decode and print!
result = tokenizer.decode(outputs[0], skip_special_tokens=True)
# Only print the output part (hide the prompt)
print(result.split("[/INST]")[-1].strip())

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
Caching is incompatible with gradient checkpointing in MistralDecoderLayer. Setting `past_key_values=None`.


Generating AI response...

{" " +  Question: Q Q Q #[[[1[INSTED: Question: Question  Question: Q Q #[[INST_ Question: Q #[INST_ Question: Question  Q #  Question: Question: Q # -  # @
 Q Q #2** ( Question: Question: Question Question: Question: Question: Q Q #  Question: Question: Question: Question: Question: Question Question: Question: Question: Question: Q #  Question: Question: Question  Q Q Q Q #[[INST: Question  Question: Question: Question Question: Question: Question: Question  Question: Question: Question: Question  Q # [ + i #  Question: Question: Question Question  Question: Question: Question: Question: Question: Question: Question: Question Question: Question: Question  Question  Question: Question Question: Question: Question: Question: Question: Q Q #  Question: Question: Question: Question: Question: Question: # []
 Q # #[] ( Question: Question: Question: Question  Question: Question: Question: Question: Q # |  Q #  Q #  Question: Question: #[[[INST IN


In [ ]:
import shutil
from google.colab import files

# Zip the folder
print("Zipping the adapter...")
shutil.make_archive("yt-custom-lora-adapter", 'zip', "yt-custom-lora-adapter")

# Download it to your computer
print("Downloading...")
files.download("yt-custom-lora-adapter.zip")

Zipping the adapter...
Downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import shutil
from google.colab import files

# Zip the folder
print("Zipping the adapter...")
shutil.make_archive("yt_lora_results", 'zip', "yt_lora_results")

# Download it to your computer
print("Downloading...")
files.download("yt_lora_results.zip")

Zipping the adapter...
Downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# 1. Clean Configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

model_id = "mistralai/Mistral-7B-Instruct-v0.2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

print("Loading clean base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

print("Injecting your custom LoRA brain...")
# 2. Attach the adapter you just trained!
model = PeftModel.from_pretrained(base_model, "yt-custom-lora-adapter")

# 3. CRITICAL: Put the model in evaluation (inference) mode!
model.eval()
base_model.config.use_cache = True

# 4. Prepare the Test Prompt
test_instruction = "You are an expert YouTube SEO strategist. Generate a viral title, description, and tags for this topic."
test_input = "Topic: How to build a modern portfolio website. Context: In this video, I will show you step by step how to build a stunning, responsive portfolio website using React, Tailwind CSS, and Neo-Brutalism design concepts to help you land your first tech job."

prompt = f"<s>[INST] {test_instruction}\n{test_input} [/INST]"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# 5. Generate the Output
print("\nGenerating AI response...\n")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=250,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

# 6. Decode and display the magic
result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("--- FINAL OUTPUT ---")
print(result.split("[/INST]")[-1].strip())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading clean base model...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Injecting your custom LoRA brain...

Generating AI response...

--- FINAL OUTPUT ---
{"title": "How to build a modern portfolio website", "description": "In this video, I will show you step by step how to build a stunning, responsive portfolio website using React, Tailwind CSS, and Neo-Brutalism design concepts to help you land your first tech job. \n\n\ud83d\udd34\ufe0f Full course: https://bit.ly/3EG2xXk\n\n\u2705\ufe0f Learn more about us:\n\ud83d\udc49\u200d\u1f46\u00a0MyCodingJourney\u200d\u1f46\ud83d\udc49\u00a0https://www.youtube.com/c/MyCodingJourney\n\n\u2705\ufe0f Learn more about our sponsors:\n\ud83d\udc49\u200d\u1f46\u00a0Solo


In [ ]:
from huggingface_hub import login

# 1. Log in to Hugging Face
login("YOUR_HF_TOKEN")

# 2. Push your custom LoRA adapter to your profile!
# Replace 'your-username' with your actual Hugging Face username
hf_username = "SujalChhajed925"
repo_name = f"{hf_username}/yt-seo-mistral-lora"

print(f"Uploading custom model to {repo_name}...")
model.push_to_hub(repo_name)
tokenizer.push_to_hub(repo_name)

print("✅ Upload complete! Your custom model is now live on Hugging Face!")

Uploading custom model to SujalChhajed925/yt-seo-mistral-lora...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   1%|1         |  557kB / 54.6MB            

README.md: 0.00B [00:00, ?B/s]

✅ Upload complete! Your custom model is now live on Hugging Face!


In [ ]:
!pip install -q -U accelerate peft transformers

In [ ]:
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from huggingface_hub import login

# 1. Log in to Hugging Face
login("YOUR_HF_TOKEN") # <-- IMPORTANT: Put your actual token here!

# 2. Wipe the GPU completely clean
torch.cuda.empty_cache()
gc.collect()

print("Loading base model strictly into the 15GB GPU VRAM...")
base_model_id = "mistralai/Mistral-7B-Instruct-v0.2"

# 3. We load directly to GPU 0. No CPU, no Disk offloading.
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map={"": 0}
)

# -----------------------------------------------------------------
# 🚨 BUG FIX FOR "UNHASHABLE TYPE: SET"
# This prevents Peft from crashing when we don't use disk offloading
if hasattr(base_model, "_no_split_modules"):
    base_model._no_split_modules = ["MistralDecoderLayer"]
if hasattr(base_model.config, "no_split_module_classes"):
    base_model.config.no_split_module_classes = ["MistralDecoderLayer"]
# -----------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(base_model_id)

print("Loading custom LoRA adapter...")
model = PeftModel.from_pretrained(base_model, "yt-custom-lora-adapter")

print("Merging weights... (Doing this in-place on the GPU)")
# The LoRA weights are small (~50MB), so this should just squeeze into the remaining VRAM!
model = model.merge_and_unload()

hf_username = "SujalChhajed925"
repo_name = f"{hf_username}/yt-seo-mistral-merged"

print(f"Uploading standalone model to {repo_name}...")
model.push_to_hub(repo_name, max_shard_size="2GB")
tokenizer.push_to_hub(repo_name)

print("✅ Merging and Upload complete! Your standalone model is ready for production.")

Loading base model strictly into the 15GB GPU VRAM...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loading custom LoRA adapter...
Merging weights... (Doing this in-place on the GPU)
Uploading standalone model to SujalChhajed925/yt-seo-mistral-merged...


Writing model shards:   0%|          | 0/8 [00:00<?, ?it/s]

...

  ...0004-of-00008.safetensors:   0%|          | 8.31MB / 1.99GB            

  ...0007-of-00008.safetensors:   0%|          | 8.31MB / 1.94GB            

  ...0005-of-00008.safetensors:   0%|          | 8.36MB / 1.94GB            

README.md: 0.00B [00:00, ?B/s]

✅ Merging and Upload complete! Your standalone model is ready for production.


In [ ]:
# Clone the library and compile it
!git clone https://github.com/ggerganov/llama.cpp
%cd llama.cpp
!make -j
!pip install -r requirements.txt
%cd ..

fatal: destination path 'llama.cpp' already exists and is not an empty directory.
/content/llama.cpp
Makefile:6: *** Build system changed:
 The Makefile build has been replaced by CMake.

 For build instructions see:
 https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md

.  Stop.
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
  Using cached transformers-4.57.6-py3-none-any.whl.metadata (43 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
Using cached transformers-4.57.6-py3-none-any.whl (12.0 MB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566

/content


In [ ]:
!pip install -q -U sentencepiece tokenizers transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.3/553.3 kB 18.5 MB/s eta 0:00:00


In [ ]:
import os
import subprocess
import sys
import shutil
from huggingface_hub import snapshot_download, HfApi, hf_hub_download

# 1. Download Model
print("1. Downloading your 14.5GB model from Hugging Face...")
model_path = snapshot_download(repo_id="SujalChhajed925/yt-seo-mistral-merged")

# 1.5 CRITICAL HOTFIX: Inject the missing tokenizer dictionary!
print("\n1.5 Patching missing tokenizer dictionary...")
try:
    original_tokenizer_path = hf_hub_download(repo_id="mistralai/Mistral-7B-Instruct-v0.2", filename="tokenizer.model")
    shutil.copy(original_tokenizer_path, os.path.join(model_path, "tokenizer.model"))
    print("✅ Tokenizer patched successfully!")
except Exception as e:
    print(f"⚠️ Could not patch tokenizer automatically: {e}")

# 2. Convert to uncompressed GGUF
print("\n2. Converting to standard GGUF format... (This takes a couple of minutes)")
convert_cmd = [sys.executable, "llama.cpp/convert_hf_to_gguf.py", model_path, "--outfile", "uncompressed.gguf"]
result_convert = subprocess.run(convert_cmd, capture_output=True, text=True)

if not os.path.exists("uncompressed.gguf"):
    print("❌ CONVERSION FAILED! Here is the hidden error log:")
    print(result_convert.stderr)
    sys.exit(1)
print("✅ Conversion to GGUF complete!")

# 3. Quantize to 4GB
print("\n3. Quantizing (Compressing) down to 4GB... (CPU working hard!)")
quantize_exe = "./llama.cpp/llama-quantize" if os.path.exists("./llama.cpp/llama-quantize") else "./llama.cpp/quantize"
quantize_cmd = [quantize_exe, "uncompressed.gguf", "custom-mistral-yt-seo-Q4_K_M.gguf", "Q4_K_M"]
result_quantize = subprocess.run(quantize_cmd, capture_output=True, text=True)

if not os.path.exists("custom-mistral-yt-seo-Q4_K_M.gguf"):
    print("❌ QUANTIZATION FAILED! Here is the hidden error log:")
    print(result_quantize.stderr)
    sys.exit(1)
print("✅ Compression down to 4GB complete!")

# 4. Upload
print("\n4. Uploading your optimized 4GB model to your profile...")
api = HfApi()
try:
    api.upload_file(
        path_or_fileobj="custom-mistral-yt-seo-Q4_K_M.gguf",
        path_in_repo="custom-mistral-yt-seo-Q4_K_M.gguf",
        repo_id="SujalChhajed925/yt-seo-mistral-merged",
        token="YOUR_HF_TOKEN" # <-- IMPORTANT: Put your token here!
    )
    print("\n✅ Upload complete! You can close Colab forever.")
except Exception as e:
    print(f"\n❌ UPLOAD FAILED: {str(e)}")

1. Downloading your 14.5GB model from Hugging Face...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]


1.5 Patching missing tokenizer dictionary...
✅ Tokenizer patched successfully!

2. Converting to standard GGUF format... (This takes a couple of minutes)
✅ Conversion to GGUF complete!

3. Quantizing (Compressing) down to 4GB... (CPU working hard!)


FileNotFoundError: [Errno 2] No such file or directory: './llama.cpp/quantize'

In [1]:
import os
import subprocess
import sys
import shutil

# --- STEP 0: Rebuild the C++ Engine ---
print("0. Setting up the C++ engine (Fixing the FileNotFoundError)...")
if not os.path.exists("llama.cpp"):
    os.system("git clone https://github.com/ggerganov/llama.cpp")
    os.system("cd llama.cpp && make -j")
os.system(f"{sys.executable} -m pip install -q -U sentencepiece tokenizers transformers huggingface_hub")

from huggingface_hub import snapshot_download, HfApi, hf_hub_download

# --- STEP 1: Download Model ---
print("\n1. Downloading your 14.5GB model from Hugging Face...")
model_path = snapshot_download(repo_id="SujalChhajed925/yt-seo-mistral-merged")

# --- STEP 1.5: Patch Tokenizer ---
print("\n1.5 Patching missing tokenizer dictionary...")
try:
    original_tokenizer_path = hf_hub_download(repo_id="mistralai/Mistral-7B-Instruct-v0.2", filename="tokenizer.model")
    shutil.copy(original_tokenizer_path, os.path.join(model_path, "tokenizer.model"))
    print("✅ Tokenizer patched successfully!")
except Exception as e:
    print(f"⚠️ Could not patch tokenizer automatically: {e}")

# --- STEP 2: Convert to GGUF ---
print("\n2. Converting to standard GGUF format...")
convert_cmd = [sys.executable, "llama.cpp/convert_hf_to_gguf.py", model_path, "--outfile", "uncompressed.gguf"]
result_convert = subprocess.run(convert_cmd, capture_output=True, text=True)

if not os.path.exists("uncompressed.gguf"):
    print("❌ CONVERSION FAILED! Here is the hidden error log:")
    print(result_convert.stderr)
    sys.exit(1)
print("✅ Conversion to GGUF complete!")

# --- STEP 3: Quantize ---
print("\n3. Quantizing (Compressing) down to 4GB... (CPU working hard!)")
quantize_exe = "./llama.cpp/llama-quantize" if os.path.exists("./llama.cpp/llama-quantize") else "./llama.cpp/quantize"
quantize_cmd = [quantize_exe, "uncompressed.gguf", "custom-mistral-yt-seo-Q4_K_M.gguf", "Q4_K_M"]
result_quantize = subprocess.run(quantize_cmd, capture_output=True, text=True)

if not os.path.exists("custom-mistral-yt-seo-Q4_K_M.gguf"):
    print("❌ QUANTIZATION FAILED! Here is the hidden error log:")
    print(result_quantize.stderr)
    sys.exit(1)
print("✅ Compression down to 4GB complete!")

# --- STEP 4: Upload ---
print("\n4. Uploading your optimized 4GB model to your profile...")
api = HfApi()
try:
    api.upload_file(
        path_or_fileobj="custom-mistral-yt-seo-Q4_K_M.gguf",
        path_in_repo="custom-mistral-yt-seo-Q4_K_M.gguf",
        repo_id="SujalChhajed925/yt-seo-mistral-merged",
        token="YOUR_HF_TOKEN" # <-- IMPORTANT: Put your token here!
    )
    print("\n✅ Upload complete! You can close Colab forever.")
except Exception as e:
    print(f"\n❌ UPLOAD FAILED: {str(e)}")

0. Setting up the C++ engine (Fixing the FileNotFoundError)...

1. Downloading your 14.5GB model from Hugging Face...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]


1.5 Patching missing tokenizer dictionary...


tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

✅ Tokenizer patched successfully!

2. Converting to standard GGUF format...
✅ Conversion to GGUF complete!

3. Quantizing (Compressing) down to 4GB... (CPU working hard!)


FileNotFoundError: [Errno 2] No such file or directory: './llama.cpp/quantize'

In [2]:
# Force start in the root directory
%cd /content

# Clean up and get fresh code
!rm -rf llama.cpp
!git clone https://github.com/ggerganov/llama.cpp

# Move inside and compile with live logs
%cd llama.cpp
!cmake -B build
!cmake --build build --config Release -j 2

/content
Cloning into 'llama.cpp'...
remote: Enumerating objects: 80438, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 80438 (delta 35), reused 26 (delta 26), pack-reused 80376 (from 3)
Receiving objects: 100% (80438/80438), 296.57 MiB | 24.67 MiB/s, done.
Resolving deltas: 100% (58131/58131), done.
/content/llama.cpp
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identific

In [3]:
# Go back to the root folder where your uncompressed.gguf is sitting
%cd /content

# Run the compression! (Using the standard CMake output path)
!./llama.cpp/build/bin/llama-quantize uncompressed.gguf custom-mistral-yt-seo-Q4_K_M.gguf Q4_K_M || ./llama.cpp/build/llama-quantize uncompressed.gguf custom-mistral-yt-seo-Q4_K_M.gguf Q4_K_M

/content
main: build = 8084 (afa6bfe4f)
main: built with GNU 11.4.0 for Linux x86_64
main: quantizing 'uncompressed.gguf' to 'custom-mistral-yt-seo-Q4_K_M.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 31 key-value pairs and 291 tensors from uncompressed.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = 23f5b32646f768cc5fd1d39aeb28bd9f074c268c
llama_model_loader: - kv   3:                           general.finetune str              = 23f5b32646f768cc5fd1d39aeb28bd9f074c268c
llama_model_loader: - kv   4:                         general.size_label str              = 7.2B
llama_model_loader: - kv   5:                

In [4]:
import os
from huggingface_hub import HfApi

if not os.path.exists("/content/custom-mistral-yt-seo-Q4_K_M.gguf"):
    print("❌ Wait! The 4GB file wasn't created. Check the logs above.")
else:
    print("✅ 4GB file found! Uploading to Hugging Face...")
    api = HfApi()
    try:
        api.upload_file(
            path_or_fileobj="/content/custom-mistral-yt-seo-Q4_K_M.gguf",
            path_in_repo="custom-mistral-yt-seo-Q4_K_M.gguf",
            repo_id="SujalChhajed925/yt-seo-mistral-merged",
            token="YOUR_HF_TOKEN" # <-- IMPORTANT: Put your token here!
        )
        print("\n🎉 UPLOAD COMPLETE! You can safely close Google Colab forever.")
    except Exception as e:
        print(f"\n❌ UPLOAD FAILED: {str(e)}")

✅ 4GB file found! Uploading to Hugging Face...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...istral-yt-seo-Q4_K_M.gguf:   0%|          |  531kB / 4.37GB            


🎉 UPLOAD COMPLETE! You can safely close Google Colab forever.
